# Structured Multimodal RAG: typed rows, visual regions, and cited evidence

## NovaTech renewal-risk investigation

A customer-success lead asks which accounts require attention. The evidence lives in a typed renewal table and a dashboard screenshot/OCR extraction. We calculate only with typed values, treat OCR as uncertain visual observation, and retain separate citations.


## Evidence-routing map

```text
Question
  |-- numeric filter / aggregation --> typed table or SQL --> row citations
  |-- visual label / chart / scan ----> OCR or vision parser -> region citations
  |-- policy / narrative -------------> text retrieval ------> passage citations
  +-- combined recommendation --------> evidence bundle -----> verified answer
```

The key design choice is the operation: an LLM must not replace a governed calculation just because the source started as text.


In [ ]:
from examples.advanced.structured_rag import (
    OcrRegion, TableRow, aggregate_with_citations, citations_are_known,
    citations_for_regions, filter_rows, search_ocr_regions, validate_table_rows,
)

rows = [
    TableRow("acme-17", {"account": "Acme", "risk_usd": 125000, "currency": "USD", "as_of": "2026-08-01"}, "renewals.csv#17"),
    TableRow("globex-03", {"account": "Globex", "risk_usd": 40000, "currency": "USD", "as_of": "2026-08-01"}, "renewals.csv#03"),
]
assert validate_table_rows(rows, {"account", "risk_usd", "currency", "as_of"}) == []


## 1 — Deterministic table calculation

Filter and aggregate in code/SQL, not in model prose. Keep the list of rows in the calculation result so a response can show exactly which records contributed.


In [ ]:
acme = filter_rows(rows, account="Acme")
summary, row_citations = aggregate_with_citations(acme, "risk_usd")
print(summary)
print(row_citations)
assert summary["sum"] == 125000
assert row_citations[0].locator == "row=acme-17"


## 2 — Schema drift and unit safety

A valid-looking row can still be unsafe: missing currency, a date in the wrong timezone, or a percentage mixed with dollars. Fail closed before aggregation, then ask for corrected data or an approved conversion policy.


In [ ]:
bad_rows = rows + [TableRow("acme-18", {"account": "Acme", "risk_usd": 90000}, "renewals.csv#18")]
errors = validate_table_rows(bad_rows, {"account", "risk_usd", "currency", "as_of"})
print(errors)
assert "missing as_of, currency" in errors[0]
assert {row.values.get("currency") for row in rows} == {"USD"}


## 3 — OCR is located, uncertain evidence

An OCR string without page and bounding box cannot be reviewed. Low-confidence regions should not enter automatic answer context. This fixture has one reliable and one unreliable reading of the same dashboard label.


In [ ]:
regions = [
    OcrRegion("dashboard-warning", "renewal-dashboard", 1, (80, 290, 760, 45), "Validate Acme migration before renewal", 0.98, "dashboard.svg"),
    OcrRegion("dashboard-noise", "renewal-dashboard", 1, (80, 200, 260, 55), "Acme migration", 0.42, "dashboard.svg"),
]
hits = search_ocr_regions("Validate Acme migration", regions, min_confidence=0.8)
visual_citations = citations_for_regions(hits)
print(hits)
print(visual_citations)
assert [hit.region_id for hit in hits] == ["dashboard-warning"]


## 4 — Compose, but do not conflate, modalities

A safe brief separates computed facts, observed visual evidence, and recommendation. It never states that an OCR observation *caused* a numeric risk unless another source establishes causality.


In [ ]:
bundle = {
    "computed": {"claim": "Acme has $125,000 in listed renewal risk.", "citations": row_citations},
    "observed": {"claim": "The dashboard contains a migration-validation warning for Acme.", "citations": visual_citations},
    "inference": "Escalate Acme for human review; the sources do not prove the warning is the cause of the risk.",
}
print(bundle)
assert citations_are_known(row_citations + visual_citations, row_ids={"acme-17"}, region_ids={"dashboard-warning"})


## 5 — Production evaluation

Score numeric correctness, unit/currency correctness, schema-valid rate, row-level/region-level citation correctness, OCR confidence calibration, modality-route accuracy, tenant isolation, and answer claim support. Test scanned PDFs, merged tables, rotated pages, chart legends, low-quality images, schema drift, and injected text in documents.


## Exercises

1. Add a EUR row and require a dated, cited conversion rate before aggregation.
2. Add a second tenant’s Acme account and enforce row/asset filtering.
3. Add an unlabeled chart axis; identify what is safe to report and what requires review.
4. Add a table claim verifier that rejects numeric prose without row citations.
5. Compare text-only RAG with this modality-aware route over ten renewal-risk questions.

Read the companion [lesson](README.md) for architecture, technology choices, and production checklist.
